In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
import spacy
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
parent_folder = Path.cwd().parent
bbc_data = pd.read_csv(parent_folder.joinpath("Supporting_Docs").joinpath("bbc_news.csv"))
type(bbc_data)

pandas.core.frame.DataFrame

In [7]:
NewData = pd.DataFrame(bbc_data['title'])

In [41]:
NewData

,title,lower,noStopwords,no_punctuations,Clean_tokenize,Raw_tokenize,Lemmatized
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[refuse, work]","[Can, I, refuse, to, work, ?]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[liz, truss, brief, world, reacts, uk, politic...","['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[rationing, energy, nothing, new, offgrid, com...","[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[hunt, superyachts, sanctioned, russian, oliga...","[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[platinum, jubilee, 70, years, queen, 70, seco...","[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, year, queen, 70, second]"
...,...,...,...,...,...,...,...
995,Dominic Raab: Third senior civil servant gives...,dominic raab: third senior civil servant gives...,dominic raab: third senior civil servant gives...,dominic raab third senior civil servant gives ...,"[dominic, raab, third, senior, civil, servant,...","[Dominic, Raab, :, Third, senior, civil, serva...","[dominic, raab, third, senior, civil, servant,..."
996,Highlights: Radacanu beats Uytvanck,highlights: radacanu beats uytvanck,highlights: radacanu beats uytvanck,highlights radacanu beats uytvanck,"[highlights, radacanu, beats, uytvanck]","[Highlights, :, Radacanu, beats, Uytvanck]","[highlight, radacanu, beat, uytvanck]"
997,In pictures: Mountain bikers descend snowy peak,in pictures: mountain bikers descend snowy peak,pictures: mountain bikers descend snowy peak,pictures mountain bikers descend snowy peak,"[pictures, mountain, bikers, descend, snowy, p...","[In, pictures, :, Mountain, bikers, descend, s...","[picture, mountain, bikers, descend, snowy, peak]"
998,"Companies must help cut living costs, says new...","companies must help cut living costs, says new...","companies must help cut living costs, says new...",companies must help cut living costs says new ...,"[companies, must, help, cut, living, costs, sa...","[Companies, must, help, cut, living, costs, ,,...","[company, must, help, cut, living, cost, say, ..."


In [12]:
NewData['lower'] = pd.Dataframe([x.lower() for x in NewData['title']])

In [16]:
en_stopwords = stopwords.words('english')
NewData['noStopwords'] = pd.DataFrame([' '.join([y for y in x.split() if y not in en_stopwords]) for x in NewData['lower']])

In [18]:
NewData['no_punctuations'] = pd.DataFrame([re.sub(r"[^\w\s]","",x) for x in NewData['noStopwords']])

In [21]:
NewData['Clean_tokenize'] = NewData.apply(lambda x : word_tokenize(x['no_punctuations']),axis = 1)

In [23]:
NewData['Raw_tokenize'] = NewData.apply(lambda x : word_tokenize(x['title']),axis = 1)

In [26]:
wnl = WordNetLemmatizer()

In [29]:
NewData['Lemmatized'] = NewData.apply(lambda x : [wnl.lemmatize(y) for y in x['Clean_tokenize']],axis = 1)

In [37]:
Raw_tokens = ' '.join(sum(NewData['Raw_tokenize'],[]))

In [35]:
Lemmatized_Tokens = ' '.join(sum(NewData['Lemmatized'],[]))

In [42]:
nlp = spacy.load('en_core_web_sm')

In [43]:
spacy_doc_raw = nlp(Raw_tokens)

In [44]:
spacy_doc_Lemmatized = nlp(Lemmatized_Tokens)

In [90]:
token_list = []
pos_list = []
for tokens in spacy_doc_Lemmatized:
    token_list.append(tokens.text)
    pos_list.append(tokens.pos_)
POS_DF_Lemmatized = pd.DataFrame({'Token' : token_list ,'POS' : pos_list})
POS_DF_Lemmatized

POS_DF_Lemm_freq = POS_DF_Lemmatized.groupby(['Token','POS']).size().reset_index(name = 'count').sort_values(by = 'count', ascending=False)
POS_DF_Lemm_verb = POS_DF_Lemm_freq[POS_DF_Lemm_freq.POS=='NOUN'][1:10]
POS_DF_Lemm_verb

,Token,POS,count
3948,world,NOUN,30
2136,man,NOUN,22
907,day,NOUN,21
3973,year,NOUN,20
1158,energy,NOUN,17
2847,record,NOUN,17
3935,woman,NOUN,16
1130,election,NOUN,16
3870,week,NOUN,16


In [78]:
token_list = []
pos_list = []
for tokens in spacy_doc_raw:
    token_list.append(tokens.text)
    pos_list.append(tokens.pos_)
POS_DF_Raw = pd.DataFrame({'Token' : token_list ,'POS' : pos_list})
POS_DF_Raw_freq = POS_DF_Raw.groupby(['Token','POS']).size().reset_index(name = 'count').sort_values(by = 'count', ascending = False)
POS_DF_Raw_freq

,Token,POS,count
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172
...,...,...,...
2304,crumbling,VERB,1
2305,crunch,PROPN,1
827,Jarrod,PROPN,1
826,Japanese,ADJ,1


In [80]:
token_list = []
ner_list = []
for tokens in spacy_doc_Lemmatized.ents:
    token_list.append(tokens.text)
    ner_list.append(tokens.label_)
NER_DF_Lemmatized = pd.DataFrame({'Token' : token_list ,'NER' : ner_list})

NER_DF_Lemm_freq = NER_DF_Lemmatized.groupby(['Token','NER']).size().reset_index(name = 'count').sort_values(by = 'count', ascending=False)
display(NER_DF_Lemm_freq)

,Token,NER,count
34,2022,CARDINAL,30
451,russian,NORP,25
223,first,ORDINAL,15
35,2022,DATE,11
450,russia,GPE,10
...,...,...,...
208,eli white,PERSON,1
207,election finland,ORG,1
206,el sistema,PERSON,1
205,el clasico,GPE,1


In [86]:
token_list = []
ner_list = []
for tokens in spacy_doc_raw.ents:
    token_list.append(tokens.text)
    ner_list.append(tokens.label_)
NER_DF_Raw = pd.DataFrame({'Tokens' : token_list , 'NER' : ner_list})
NER_DF_Raw_freq = NER_DF_Raw.groupby(['Tokens','NER']).size().reset_index(name = 'count').sort_values(by = 'count', ascending=False)
NER_DF_Raw_freq

,Tokens,NER,count
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
...,...,...,...
405,Georgia Taylor-Brown,PERSON,1
406,Geraint Thomas,PERSON,1
409,Ghislaine Maxwell,PERSON,1
410,Gianluigi Lentini,PERSON,1
